# TNB Frame Animation

Edit `r_sym` in the cell below to define your parametric curve, then run all cells.

The animation will render and play inline.

In [ ]:
from manim import *
import sympy as s
import numpy as np
from sympy import symbols, sqrt, lambdify, sin, cos, tan, exp, log, pi

t = symbols('t')

In [ ]:
# ── Edit your curve here ──────────────────────────────────────────
r_sym = s.Matrix([2*cos(3*t), 2*sin(t), 2*cos(2*t)])

# t range for the animation
t_start = 0
t_end   = 2 * float(pi)
# ─────────────────────────────────────────────────────────────────

In [ ]:
%%manim -qm TNBFrame

class TNBFrame(ThreeDScene):
    def construct(self):
        def TNB_get(func):
            v_sym = func.diff(t)
            speed_sym = sqrt(v_sym.dot(v_sym))
            T_sym = v_sym / speed_sym
            N_sym = T_sym.diff(t) / sqrt(T_sym.diff(t).dot(T_sym.diff(t)))
            B_sym = T_sym.cross(N_sym)
            T = lambdify(t, T_sym, 'numpy')
            N = lambdify(t, N_sym, 'numpy')
            B = lambdify(t, B_sym, 'numpy')
            return T, N, B

        axes = ThreeDAxes(x_length=12, y_length=10, z_length=8)
        self.move_camera(phi=PI/3, theta=PI/4)
        self.add(axes)
        self.wait()

        r_temp = lambdify(t, r_sym, 'numpy')
        r_plot = lambda t: np.squeeze(r_temp(t))

        C  = ParametricFunction(r_plot, t_range=[t_start, t_end], color=RED)
        t0 = MathTex(r'\vec{r}(t)=', color=RED)
        T_l = MathTex(r'\vec{T}', color=GREEN)
        N_l = MathTex(r'\vec{N}', color=BLUE)
        B_l = MathTex(r'\vec{B}', color=PURPLE)
        TNB_l = VGroup(T_l, N_l, B_l)

        self.add_fixed_in_frame_mobjects(t0.shift(3*UP + 3*RIGHT).scale(0.75))
        self.play(Create(C), run_time=5)
        self.wait()

        tracker = ValueTracker(t_start)
        T_f, N_f, B_f = TNB_get(r_sym)
        T = lambda t: np.squeeze(T_f(t))
        N = lambda t: np.squeeze(N_f(t))
        B = lambda t: np.squeeze(B_f(t))

        T_p = Vector(T(t_start), color=GREEN).shift(r_plot(t_start))
        T_p.add_updater(lambda m: m.become(
            Vector(T(tracker.get_value()), color=GREEN).shift(r_plot(tracker.get_value()))))
        N_p = Vector(N(t_start), color=BLUE).shift(r_plot(t_start))
        N_p.add_updater(lambda m: m.become(
            Vector(N(tracker.get_value()), color=BLUE).shift(r_plot(tracker.get_value()))))
        B_p = Vector(B(t_start), color=PURPLE).shift(r_plot(t_start))
        B_p.add_updater(lambda m: m.become(
            Vector(B(tracker.get_value()), color=PURPLE).shift(r_plot(tracker.get_value()))))

        self.add(VGroup(T_p, N_p, B_p))
        self.add_fixed_in_frame_mobjects(TNB_l.arrange(DOWN).next_to(t0, 2*RIGHT))
        self.wait()
        self.play(tracker.animate.set_value(t_end), run_time=10, rate_func=linear)
        self.wait()